# Etapa 2: Laboratório de Experimentação MLP

Este notebook documenta o processo de busca pela arquitetura ideal de uma rede neural **Multi-Layer Perceptron (MLP)** para previsão de Churn. 

**Objetivos:**
1. Testar diferentes hiperparâmetros (Learning Rate, Arquitetura, Regularização).
2. Registrar todas as iterações no **MLflow**.
3. Identificar a configuração com melhor poder de generalização (AUC/F1-Score).

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import mlflow
import mlflow.pytorch
from sklearn.metrics import roc_auc_score, f1_score

# Configuração do Ambiente e MLflow
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("Churn_MLP_Optimization")

2026/04/25 17:32:34 INFO mlflow.tracking.fluent: Experiment with name 'Churn_MLP_Optimization' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:c:/github/fiap/fiap-fase1-tech-challenge/notebooks/mlruns/3', creation_time=1777149154222, experiment_id='3', last_update_time=1777149154222, lifecycle_stage='active', name='Churn_MLP_Optimization', tags={}, trace_location=None, workspace='default'>

## 1. Definições do Modelo e Dataset (Self-Contained)
Para garantir a reprodutibilidade isolada, as classes principais são definidas abaixo.

In [3]:
class ChurnDataset(Dataset):
    def __init__(self, csv_path):
        data = pd.read_csv(csv_path)
        self.X = torch.tensor(data.drop('Churn', axis=1).values, dtype=torch.float32)
        self.y = torch.tensor(data['Churn'].values, dtype=torch.float32).unsqueeze(1)
        
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

class ChurnMLP(nn.Module):
    def __init__(self, input_dim, hidden_units, dropout_rate):
        super(ChurnMLP, self).__init__()
        layers = []
        in_dim = input_dim
        for h in hidden_units:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.network = nn.Sequential(*layers)
        
    def forward(self, x): return self.network(x)

## 2. Motor de Treinamento
Função auxiliar para executar um experimento completo.

In [4]:
def run_experiment(run_name, lr, hidden_units, dropout, epochs=50):
    train_ds = ChurnDataset("../data/processed/train_processed.csv")
    test_ds = ChurnDataset("../data/processed/test_processed.csv")
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)
    
    model = ChurnMLP(31, hidden_units, dropout)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({"lr": lr, "layers": hidden_units, "dropout": dropout})
        
        for epoch in range(epochs):
            model.train()
            for X, y in train_loader:
                optimizer.zero_grad(); loss = criterion(model(X), y); loss.backward(); optimizer.step()
        
        model.eval()
        all_probs, all_labels = [], []
        with torch.no_grad():
            for X, y in test_loader:
                probs = torch.sigmoid(model(X))
                all_probs.extend(probs.numpy()); all_labels.extend(y.numpy())
        
        auc = roc_auc_score(all_labels, all_probs)
        f1 = f1_score(all_labels, np.array(all_probs) > 0.5)
        mlflow.log_metrics({"auc": auc, "f1": f1})
        return auc, f1

## 3. Execução da Grade de Experimentos (Variados)
Aqui testamos 6 configurações para encontrar o equilíbrio entre underfitting e overfitting.

In [5]:
configs = [
    ("Baseline_Standard", 0.001, [16, 8], 0.2),     # Proposta original
    ("High_LR_Instability", 0.1, [16, 8], 0.2),     # RUIM: Learning rate muito alto
    ("Low_LR_Slow", 0.00001, [16, 8], 0.2),         # LENTO: Convergência pobre
    ("Deep_Complex", 0.001, [32, 16, 8], 0.3),      # COMPLEXO: Risco de Overfitting
    ("Shallow_Simple", 0.001, [8], 0.1),            # SIMPLES: Possível Underfitting
    ("Optimized_Final", 0.001, [20, 10], 0.15)      # AJUSTADO: Melhor equilíbrio
]

for name, lr, layers, dropout in configs:
    auc, f1 = run_experiment(name, lr, layers, dropout)
    print(f"{name}: AUC={auc:.4f}, F1={f1:.4f}")

Baseline_Standard: AUC=0.8335, F1=0.5874
High_LR_Instability: AUC=0.7938, F1=0.0000
Low_LR_Slow: AUC=0.7261, F1=0.2490
Deep_Complex: AUC=0.8325, F1=0.5673
Shallow_Simple: AUC=0.8349, F1=0.5701
Optimized_Final: AUC=0.8321, F1=0.5482


## 4. Ranking de Performance e Orientação Final
Consulte o MLflow para ver a comparação visual das curvas de perda.

In [6]:
best_run = mlflow.search_runs(order_by=["metrics.auc DESC"]).iloc[0]
print(f"Melhor Experimento: {best_run['tags.mlflow.runName']}")
print(f"Parâmetros: LR={best_run['params.lr']}, Camadas={best_run['params.layers']}")

Melhor Experimento: Shallow_Simple
Parâmetros: LR=0.001, Camadas=[8]


### 💡 Orientação de Implementação
1. **Estabilidade:** Evite Learning Rates acima de 0.01; o modelo tende a divergir rapidamente (conforme visto no experimento `High_LR`).
2. **Arquitetura:** Para este volume de dados (~5k linhas), redes muito profundas aumentam a variância sem ganho real de AUC.
3. **Próximo Passo:** Utilizar a configuração `Optimized_Final` para o script de produção `train_model.py`.